# Function declarations

Every function, class and constant the project uses is defined here, and nothing is
executed. `data_processing.ipynb` and `main.ipynb` each load this notebook with
`%run ./function_declaration.ipynb` and then do all the running.

The sections follow the order of the pipeline:

1. **Data loading**: read the raw NOAA/GLOS files and merge them into one monthly table
2. **Models**: persistence, seasonal climatology, water balance, lake model, gradient boosting, hybrid, physics-guided CNN
3. **Evaluation**: rolling-origin backtest, RMSE, bootstrap comparison
4. **Figures**: the five report figures

## 0. Imports and paths

In [1]:
import copy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.optimize import least_squares
from sklearn.ensemble import GradientBoostingRegressor
from torch import nn

In [3]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "figures"

## 1. Data loading

Read the raw NOAA/GLOS files and merge them into one monthly table.

Raw files in `data/raw/` are never modified. Everything here is a pure
read → reshape → merge.

In [4]:
AREA_MIC_KM2 = 57_800.0
AREA_HUR_KM2 = 59_600.0
AREA_COMBINED_M2 = (AREA_MIC_KM2 + AREA_HUR_KM2) * 1e6

START = "1950-01"

**Precipitation and evaporation files.** One row per year with twelve monthly
columns. We want one long series with one value per month. Values are in mm.

In [5]:
def read_depth_table(name):
    wide = pd.read_csv(RAW / name, comment="#", skipinitialspace=True)
    wide.columns = range(1, 13)
    long = wide.stack()
    long.index = pd.PeriodIndex(
        [f"{year}-{month:02d}" for year, month in long.index], freq="M"
    )
    return long

**Runoff files.** Already one row per month; keep the flow and the share of the basin that was gauged.

In [6]:
def read_runoff(name):
    df = pd.read_csv(RAW / name, sep=r"\s*,\s*", engine="python")
    df.index = pd.PeriodIndex(df["YYYY-MM"], freq="M")
    return df[["flow_cms", "fraction_gauged"]]

**Lake levels.** Monthly lake-wide mean level from GLOS. Levels are in metres (IGLD 1985 datum); the second row of the file holds units, not data.

In [7]:
def read_levels():
    df = pd.read_csv(RAW / "levels_michigan_huron.csv", skiprows=[1])
    idx = pd.PeriodIndex(
        [f"{y}-{m:02d}" for y, m in zip(df["Year"], df["Month"])], freq="M"
    )
    return pd.Series(df["Water_Level_m"].to_numpy(), index=idx)

**Runoff as a depth.** Runoff is a flow (m³/s). Multiply by the seconds in the month
to get a volume, then spread it over the lake surface to get millimetres, the same
unit as precipitation and evaporation.

In [8]:
def runoff_to_depth_mm(flow_cms):
    seconds_in_month = flow_cms.index.days_in_month * 86_400
    return flow_cms * seconds_in_month / AREA_COMBINED_M2 * 1000.0

**Merge everything.** Michigan and Huron are treated as one lake. Precipitation and
evaporation are depths, so they are averaged by lake area; runoff is a volume, so the
two basins are added. The target `dlevel_mm` is the month-to-month change in level.

In [9]:
def build_monthly():
    w_mic = AREA_MIC_KM2 / (AREA_MIC_KM2 + AREA_HUR_KM2)
    w_hur = 1.0 - w_mic

    # Precip and evap are depths over each lake -> area-weighted average.
    precip = w_mic * read_depth_table("prc_mic_lake.csv") + w_hur * read_depth_table(
        "prc_hur_lake.csv"
    )
    evap = w_mic * read_depth_table("evaporation_mic.csv") + w_hur * read_depth_table(
        "evaporation_hur.csv"
    )

    # Runoff is a volume flux -> add the two basins, then convert to depth.
    ro_mic = read_runoff("runoff_mic_arm.csv")
    ro_hur = read_runoff("runoff_hur_arm.csv")
    runoff = runoff_to_depth_mm(ro_mic["flow_cms"] + ro_hur["flow_cms"])

    df = pd.DataFrame(
        {
            "level_m": read_levels(),
            "precip_mm": precip,
            "evap_mm": evap,
            "runoff_mm": runoff,
            "frac_gauged_mic": ro_mic["fraction_gauged"],
            "frac_gauged_hur": ro_hur["fraction_gauged"],
        }
    )
    df["nbs_mm"] = df["precip_mm"] - df["evap_mm"] + df["runoff_mm"]
    # Target: change in the monthly mean level from the previous month, in mm.
    df["dlevel_mm"] = df["level_m"].diff() * 1000.0

    df = df.loc[START:]
    # Keep only the span where every series is observed.
    complete = df[["level_m", "precip_mm", "evap_mm", "runoff_mm"]].notna().all(axis=1)
    df = df.loc[: complete[complete].index.max()]
    assert df.notna().all().all(), "unexpected gap inside the analysis period"
    df.index.name = "month"
    return df

**Over-land weather.** Daily over-land air temperature and precipitation for each basin (GLERL), turned into monthly totals for the snowpack model: snowfall (precipitation on days with a mean temperature below 0 °C), rainfall (the rest) and positive degree-days (the sum of the daily mean temperatures above 0 °C).

In [ ]:
LAND = [f"{v}_{b}" for b in ("mic", "hur") for v in ("snow", "rain", "pdd")]


def read_land_weather(lake):
    d = pd.read_csv(RAW / f"subdata_{lake}_land.csv", comment="#", sep=r"\s*,\s*", engine="python")
    t, month = (d["AirTempMin"] + d["AirTempMax"]) / 2, pd.PeriodIndex(d["Date"], freq="M")
    return pd.DataFrame({f"snow_{lake}": d["Precip"].where(t < 0, 0.0), f"rain_{lake}": d["Precip"].where(t >= 0, 0.0),
                         f"pdd_{lake}": t.clip(lower=0)}).groupby(month).sum()

## 2. Models

Every model answers the same question: standing at the end of month *t* (the
"origin"), knowing levels and P/E/R up to and including month *t*, what is the level
change `L[t+h] - L[t]` for *h* = 1..6 months ahead?

All models share one interface:

```python
model.fit(train)                     # train: monthly DataFrame, rows before the test year
model.predict(df, origins, horizons) # -> array (n_origins, n_horizons), cumulative change in mm
```

`df` passed to `predict` may contain future rows; a model must only use rows up to
each origin (the perfect-forcing runs are the deliberate exception).

In [10]:
COMPONENTS = ["precip_mm", "evap_mm", "runoff_mm"]
LEVEL_REF_M = 176.5  # only centres the level term; has no effect on forecasts

**Month of the year.** A small helper: give it a monthly index and it returns the calendar
month number (1–12) of every row. It is used wherever something is averaged by calendar month.

In [11]:
def month_of(index):
    return index.month.to_numpy()

**Persistence.** The simplest possible forecast: the level stays where it is.

What each part does:

- `fit(train)`: nothing to learn, so it just returns the model. It has to exist because
  the backtest calls `fit` on every model.
- `predict(df, origins, horizons)`: returns zeros, one row per origin and one column per
  horizon, meaning "no change".

In [12]:
class Persistence:
    def fit(self, train):
        return self

    def predict(self, df, origins, horizons):
        return np.zeros((len(origins), len(horizons)))

**Seasonal climatology.** The lake rises in spring and falls in autumn; forecast the average change for each calendar month.

What each part does:

- `fit(train)`: averages the month-to-month level change for each calendar month over
  the training years, giving twelve numbers.
- `predict(df, origins, horizons)`: for each origin, adds up those averages for the
  coming months. The forecast at horizon *h* is the sum of the first *h* of them.

In [13]:
class SeasonalClimatology:
    def fit(self, train):
        self.mean_change_ = train["dlevel_mm"].groupby(month_of(train.index)).mean()
        return self

    def predict(self, df, origins, horizons):
        steps = self.mean_change_.to_numpy()[(origins.month.to_numpy()[:, None] + np.arange(max(horizons))) % 12]
        return np.cumsum(steps, axis=1)[:, np.array(horizons) - 1]

**Water balance.** A regression shaped like the water budget: level change = supply
(precipitation, evaporation, runoff) + a monthly constant + a term that lets outflow
grow when the lake is high. It is stepped forward one month at a time.

```
dL[t] = a[month] + sum_c (b0_c * X_c[t] + b1_c * X_c[t-1]) + g * (L[t-1] - ref)
```

for *c* in precipitation, evaporation, runoff.

- Two lags per component because levels are monthly *means*: the change between two
  monthly means straddles two months of supply.
- `a[month] + g * level` stands in for the connecting-channel flows (St. Marys in,
  St. Clair out), which we have no data for; outflow rises with level.
- To forecast, future P/E/R are unknown, so each is replaced by its calendar-month
  climatology plus a decaying share of the latest anomaly (AR(1)).
- With `perfect_forcing=True` the observed future P/E/R are used instead. That is not a
  real forecast, but an upper bound on what this model could do with a perfect weather
  forecast.
- `coefficients()` returns the fitted coefficients as a labelled table.

What each part does:

- `__init__(perfect_forcing)`: remembers whether to use the observed future P/E/R.
- `_design(...)`: builds the regression table: twelve month indicator columns, this
  month's P/E/R, last month's P/E/R, and the previous level minus the reference level.
- `fit(train)`: solves for the coefficients by least squares, then learns the forcing model.
- `_fit_forcing(train)`: the forcing model: the calendar-month average of P/E/R and how
  strongly an anomaly carries over to the next month (`phi_`). The lake model reuses it.
- `_forcing(df, origins, k, observed)`: P/E/R for the month `k` steps after each origin:
  the observed values, or the climatology plus the origin's anomaly shrunk by `phi_**k`.
- `predict(df, origins, horizons)`: steps all origins forward together, one month at a time,
  feeding its own predicted level back in and adding up the changes.

In [14]:
class WaterBalance:
    def __init__(self, perfect_forcing=False):
        self.perfect_forcing = perfect_forcing

    def _design(self, month, x_now, x_prev, level_prev):
        dummies = np.zeros((len(month), 12))
        dummies[np.arange(len(month)), month - 1] = 1.0
        return np.column_stack([dummies, x_now, x_prev, level_prev - LEVEL_REF_M])

    def fit(self, train):
        x = train[COMPONENTS].to_numpy()
        design = self._design(month_of(train.index)[1:], x[1:], x[:-1], train["level_m"].to_numpy()[:-1])
        self.coef_, *_ = np.linalg.lstsq(design, train["dlevel_mm"].to_numpy()[1:], rcond=None)
        return self._fit_forcing(train)

    def _fit_forcing(self, train):
        self.clim_ = train[COMPONENTS].groupby(month_of(train.index)).mean()
        anom = train[COMPONENTS].to_numpy() - self.clim_.loc[month_of(train.index)].to_numpy()
        self.phi_ = np.array([np.corrcoef(a[1:], a[:-1])[0, 1] for a in anom.T])
        return self

    def _forcing(self, df, origins, k, observed):
        if observed:
            return df.loc[origins + k, COMPONENTS].to_numpy()
        anom = df.loc[origins, COMPONENTS].to_numpy() - self.clim_.loc[origins.month].to_numpy()
        return self.clim_.loc[(origins + k).month].to_numpy() + self.phi_**k * anom

    def predict(self, df, origins, horizons):
        level, x_prev, cum, path = df.loc[origins, "level_m"].to_numpy(), df.loc[origins, COMPONENTS].to_numpy(), 0, []
        for k in range(1, max(horizons) + 1):
            x_now = self._forcing(df, origins, k, self.perfect_forcing)
            cum = cum + self._design((origins + k).month.to_numpy(), x_now, x_prev, level + cum / 1000) @ self.coef_
            path.append(cum)
            x_prev = x_now
        return np.column_stack(path)[:, np.array(horizons) - 1]

    def coefficients(self):
        names = [f"month_{m}" for m in range(1, 13)]
        names += [f"{c}_lag0" for c in COMPONENTS] + [f"{c}_lag1" for c in COMPONENTS]
        names += ["level_m"]
        return pd.Series(self.coef_, index=names)

**Lake model.** A more physical version of the water balance. Instead of a regression, the
lake is a storage that fills and drains continuously, and its outflow follows a rating curve:

```
dL/dt = mP * P - mE * E + mR * R + n[month] - c * (((L - z0) / d) ** 1.5 - 1),   z0 = LEVEL_REF_M - d
```

with every term in mm per month over the lake.

- `mP`, `mE`, `mR` scale the P/E/R estimates, which are themselves model estimates with biases.
- `n[month]` is the net connecting-channel flow at the reference level (St. Marys in, St. Clair
  and the Chicago diversion out), one value per calendar month because these flows have a
  seasonal cycle (ice slows the St. Clair River in winter, for example).
- The last term is the extra outflow when the lake is above the reference level. Flow over a
  weir grows with the depth of water above the sill to the power 1.5; `z0` is that effective
  sill and `c` the outflow at the reference level. The channel flows are not observed, so
  `n[month]` and `c` are only fitted together: the data pin down how strongly the outflow
  responds to the level, not the size of the real outflow.
- The level is stepped forward 10 times per month, and each month's forecast is the average of
  its steps, because the observed levels are monthly means. A run starts from the observed mean
  of the origin month (its middle) and first steps half a month to the end of that month.
- The 17 parameters are fitted by non-linear least squares to the observed 1–6-month level
  changes of the training years, driven by the observed P/E/R, so they describe the lake and
  not the weather forecast.
- To forecast, the weather is filled in exactly as in the water-balance model (climatology plus
  AR(1)); `perfect_forcing=True` uses the observed weather instead.
- `lake_supply` and `lake_route` use only arithmetic, so the same code runs on NumPy arrays here
  and on PyTorch tensors in the CNN below.

What each part does:

- `lake_supply(p, x, month)`: the water entering the lake, `mP*P - mE*E + mR*R + n[month]`.
- `lake_route(p, level, supply)`: steps the level through the months and returns each month's
  mean level.
- `fit(train)`: learns the forcing model, then fits the parameters `p_`.
- `_inputs(df, origins, n_months, observed)`: the starting level, the P/E/R of the origin month
  and of the next `n_months`, and their calendar months.
- `_changes(p, ...)`: runs the lake and returns the level change from the origin in mm.
- `predict(df, origins, horizons)`: `_changes` with the fitted parameters.

In [ ]:
def lake_supply(p, x, month):
    return p[0] * x[..., 0] - p[1] * x[..., 1] + p[2] * x[..., 2] + p[3:15][month - 1]


def lake_route(p, level, supply, substeps=10):
    def rate(level, s):
        return s - p[15] * (((level - LEVEL_REF_M + p[16]) / p[16]) ** 1.5 - 1)

    level = level + rate(level, supply[:, 0]) / 2000  # half a month: monthly mean -> end of month
    means = []
    for k in range(1, supply.shape[1]):
        total = 0
        for _ in range(substeps):
            level = level + rate(level, supply[:, k]) / (1000 * substeps)
            total = total + level / substeps
        means.append(total)
    return means


class LakeModel(WaterBalance):
    # p = [mP, mE, mR, n[1..12], c, d]; d >= 1.5 m keeps the sill z0 below the 2013 record low
    bounds = (np.r_[0, 0, 0, [-np.inf] * 12, 0, 1.5], np.r_[3, 3, 3, [np.inf] * 12, np.inf, 50])

    def fit(self, train):
        self._fit_forcing(train)
        origins = train.index[: -max(HORIZONS)]
        y = np.column_stack([_future_change(train, h).loc[origins] for h in HORIZONS])
        args = self._inputs(train, origins, max(HORIZONS), observed=True)
        # Start: no scaling, n = average budget gap per month, slope 1.5*c/d = 24 mm/month per m (as the water balance)
        channel = (train["dlevel_mm"] - train["nbs_mm"]).groupby(month_of(train.index)).mean()
        self.p_ = least_squares(lambda p: (self._changes(p, *args) - y).ravel(),
                                np.r_[1, 1, 1, channel, 80, 5], bounds=self.bounds).x
        return self

    def _inputs(self, df, origins, n_months, observed):
        x = np.stack([df.loc[origins, COMPONENTS].to_numpy()]
                     + [self._forcing(df, origins, k, observed) for k in range(1, n_months + 1)], axis=1)
        month = np.column_stack([(origins + k).month for k in range(n_months + 1)])
        return df.loc[origins, "level_m"].to_numpy(), x, month

    def _changes(self, p, level, x, month):
        return (np.column_stack(lake_route(p, level, lake_supply(p, x, month))) - level[:, None]) * 1000

    def predict(self, df, origins, horizons):
        return self._changes(self.p_, *self._inputs(df, origins, max(horizons), self.perfect_forcing))[:, np.array(horizons) - 1]

**Snowpack and basin storage.** The water-balance and lake models fill in future runoff with an AR(1) forecast, which cannot know how much snow is lying on the land. This model adds that physics for the two basins:

- Snowfall builds a snowpack, which melts by `ddf` mm for every positive degree-day.
- Rain and meltwater fill one basin store (scaled by `a_mic` and `a_hur` to mm over the lake), evapotranspiration removes `e` mm per degree-day, and the store releases a share `k` of its water as runoff each month.
- At an origin, the snowpack comes from simulating the record up to that month, and the store is set from that month's observed runoff (runoff = `k` x store).
- The five parameters are fitted by least squares to the observed 1-6-month runoff of the training years, driven by the observed weather.
- Only the snowpack *anomaly* is used: the runoff forecast is the AR(1) forecast plus the difference between the model run from this year's snowpack and from the average snowpack of that calendar month, both with average weather. A version that replaced the AR(1) forecast entirely, and one without evapotranspiration, were worse than AR(1) in checks on 1950-1979, so they were dropped before any test year was used.

`SnowForcing` swaps this runoff forecast into a model's weather forecast; `SnowWaterBalance` and `SnowLakeModel` are the water-balance and lake models with it.

In [ ]:
class SnowRunoff:
    # p = [ddf (mm of melt per degree-day), k (share of basin storage released per month),
    #      e (evapotranspiration per degree-day), a_mic, a_hur (scale of the water reaching each basin)]
    bounds = ([0, 0.01, 0, 0, 0], [20, 1, 20, 5, 5])

    def fit(self, train):
        self.clim_, self.end_ = train[LAND].groupby(month_of(train.index)).mean(), train.index[-1]
        origins = train.index[12 : -max(HORIZONS)]
        y = np.column_stack([train["runoff_mm"].shift(-h).loc[origins] for h in HORIZONS])
        self.p_ = least_squares(lambda p: (self._run(p, train, origins, max(HORIZONS), True) - y).ravel(),
                                [3, 0.5, 0.3, 0.3, 0.3], bounds=self.bounds).x
        return self

    def _step(self, p, swe, s, snow, rain, pdd):
        swe = swe + snow
        melt = np.minimum(swe, p[0] * pdd)
        s = np.maximum(s + (rain + melt) @ p[3:] - p[2] * pdd.mean(axis=-1), 0)
        return swe - melt, (1 - p[1]) * s, p[1] * s

    def snowpack(self, df, p=None):  # simulated snowpack of both basins at the end of every month (mm over the land)
        p, swe, s, out = self.p_ if p is None else p, np.zeros(2), 0.0, []
        for snow, rain, pdd in df[LAND].to_numpy().reshape(-1, 2, 3).transpose(0, 2, 1):
            swe, s, _ = self._step(p, swe, s, snow, rain, pdd)
            out.append(swe)
        return pd.DataFrame(out, index=df.index, columns=["mic", "hur"])

    def _run(self, p, df, origins, n_months, observed, normal_snow=False):
        pack = self.snowpack(df, p)
        swe = pack.loc[origins].to_numpy()
        if normal_snow:  # the average snowpack of the origin's calendar month instead of this year's
            past = pack.loc[: self.end_]
            swe = past.groupby(month_of(past.index)).mean().loc[origins.month].to_numpy()
        s, out = df.loc[origins, "runoff_mm"].to_numpy() * (1 - p[1]) / p[1], []  # storage implied by this month's runoff
        for k in range(1, n_months + 1):
            land = (df.loc[origins + k, LAND] if observed else self.clim_.loc[(origins + k).month]).to_numpy()
            swe, s, r = self._step(p, swe, s, *land.reshape(-1, 2, 3).transpose(2, 0, 1))
            out.append(r)
        return np.column_stack(out)

    def forecast(self, df, origins, n_months):  # extra runoff caused by an unusual snowpack
        return self._run(self.p_, df, origins, n_months, False) - self._run(self.p_, df, origins, n_months, False, True)


class SnowForcing:
    def fit(self, train):
        self.snow_ = SnowRunoff().fit(train)
        return super().fit(train)

    def _forcing(self, df, origins, k, observed):
        x = super()._forcing(df, origins, k, observed)
        if not observed:
            x[:, 2] += self.snow_.forecast(df, origins, k)[:, -1]
        return x


class SnowWaterBalance(SnowForcing, WaterBalance):
    pass


class SnowLakeModel(SnowForcing, LakeModel):
    pass

**Features for the ML model.** Everything the model is allowed to know at the end of
a month: the current level, the last three months of level change and supply,
accumulated supply over 3–12 months, and the time of year. Row *t* uses data up to month *t* only.

In [15]:
def make_features(df):
    f = pd.DataFrame(index=df.index)
    f["level"] = df["level_m"] - LEVEL_REF_M
    for lag in range(3):
        f[f"dlevel_lag{lag}"] = df["dlevel_mm"].shift(lag)
        for c in COMPONENTS:
            f[f"{c[:-3]}_lag{lag}"] = df[c].shift(lag)
    for window in (3, 6, 12):
        f[f"nbs_sum{window}"] = df["nbs_mm"].rolling(window).sum()
    angle = 2 * np.pi * month_of(df.index) / 12
    f["month_sin"] = np.sin(angle)
    f["month_cos"] = np.cos(angle)
    return f

**Feature groups.** The features above bundled into seven named groups, so Figure 4 can
shuffle one whole group at a time.

In [16]:
# Feature groups used for the permutation-importance figure.
FEATURE_GROUPS = {
    "Season (month)": ["month_sin", "month_cos"],
    "Current level": ["level"],
    "Recent level change": [f"dlevel_lag{k}" for k in range(3)],
    "Precipitation": [f"precip_lag{k}" for k in range(3)],
    "Evaporation": [f"evap_lag{k}" for k in range(3)],
    "Runoff": [f"runoff_lag{k}" for k in range(3)],
    "Accumulated NBS (3-12 mo)": ["nbs_sum3", "nbs_sum6", "nbs_sum12"],
}

**Training target.** The level change from month *t* to month *t+h*, in mm, lined up on
month *t*. It is empty (NaN) where *t+h* runs past the end of the data.

In [17]:
def _future_change(df, h):
    return (df["level_m"].shift(-h) - df["level_m"]) * 1000.0

**Gradient boosting.** One tree model per horizon, predicting the level change
directly. With `base=WaterBalance()` it becomes the hybrid: the trees only learn what
the water-balance model got wrong.

Hyperparameters were fixed in advance (small, slow-learning, shallow trees for a few
hundred training rows) and were not tuned on the test years.

What each part does:

- `params`: the fixed tree settings.
- `__init__(base)`: remembers the optional base model (none for plain gradient boosting,
  `WaterBalance()` for the hybrid).
- `fit(train)`: builds the features, fits the base model if there is one, then trains one
  tree model for each horizon 1–6 on the rows where features and target both exist. With
  a base model, the target is what the base model got wrong.
- `predict(df, origins, horizons)`: looks up the features at each origin and makes one
  prediction per horizon. With a base model, its forecast is added back on.

In [18]:
class GradientBoosting:
    params = dict(n_estimators=300, learning_rate=0.03, max_depth=3,
                  subsample=0.8, min_samples_leaf=5, random_state=0)

    def __init__(self, base=None):
        self.base = base

    def _features(self, df):
        return make_features(df)

    def fit(self, train):
        self.models_ = {}
        feats = self._features(train)
        ok = feats.notna().all(axis=1)
        if self.base is not None:
            self.base.fit(train)
        for h in range(1, 7):
            y = _future_change(train, h)
            rows = ok & y.notna()
            origins = train.index[rows]
            target = y[rows].to_numpy()
            if self.base is not None:
                target = target - self.base.predict(train, origins, [h])[:, 0]
            self.models_[h] = GradientBoostingRegressor(**self.params).fit(
                feats.loc[origins], target
            )
        return self

    def predict(self, df, origins, horizons):
        # Features are backward-looking, so computing them on the full table
        # does not leak future information into a given origin's row.
        feats = self._features(df).loc[origins]
        out = np.column_stack([self.models_[h].predict(feats) for h in horizons])
        if self.base is not None:
            out = out + self.base.predict(df, origins, horizons)
        return out

**Gradient boosting + snowpack.** The same gradient boosting, also given the simulated snowpack of both basins at the origin (from a `SnowRunoff` model fitted on the training years) as two extra features. It tests whether the machine learning model gains from the same physics. A check on 1950-1979 showed it improving on plain gradient boosting at three to six months before any test year was used.

In [ ]:
class SnowGradientBoosting(GradientBoosting):
    def fit(self, train):
        self.snow_ = SnowRunoff().fit(train)
        return super().fit(train)

    def _features(self, df):
        return make_features(df).join(self.snow_.snowpack(df).add_prefix("snowpack_"))

**Physics-guided CNN.** A convolutional neural network that never predicts the level itself.
It reads the last 24 months of P, E, R, level change, level and season, and corrects the
weather forecast for the next six months. The lake model above then turns that weather into
levels, so every forecast obeys the water balance by construction.

- The weather it corrects is the lake model's own forecast (climatology plus AR(1)). The last
  layer starts at zero, so before any training the CNN gives exactly the lake model's forecast.
- Network: two 1-D convolutions (8 filters, width 3, ReLU) and one linear layer, giving a
  correction to P, E and R in each of the next six months, in units of that component's typical
  monthly anomaly.
- Loss: the level error at 1–6 months plus the weather error, each divided by its own spread.
  The weather term keeps the corrected P/E/R believable instead of bent to fit the levels.
- The lake parameters are fitted first and then held fixed; the gradients flow back through
  `lake_route` into the network.
- Early stopping: the network trains on the first 80% of the training months and keeps the
  epoch that does best on the last 20%. If no epoch helps, it stays equal to the lake model.
- The settings (24-month window, 8 filters, learning rate 0.001, weight decay 0.01, at most
  300 epochs) were fixed using 1950–1979 only, never the test years.

What each part does:

- `fit(train)`: fits the lake model, builds the training windows and targets, and trains the
  network.
- `_channels(df)`: the seven input series.
- `_windows(df, origins)`: the standardised 24 months ending at each origin, shaped
  (origins, channels, months) for the convolutions.
- `_forward(...)`: corrected weather, then the lake model, then the level change in mm.
- `predict(df, origins, horizons)`: `_forward` without gradients.

In [ ]:
class PhysicsCNN:
    window, epochs, lr = 24, 300, 1e-3

    def fit(self, train):
        torch.manual_seed(0)
        self.lake_ = LakeModel().fit(train)
        channels = self._channels(train)
        self.mu_, self.sd_ = channels.mean(), channels.std()
        anom = train[COMPONENTS] - self.lake_.clim_.loc[month_of(train.index)].to_numpy()
        self.xsd_ = torch.tensor(anom.std().to_numpy())
        origins = train.index[self.window - 1 : -max(HORIZONS)]
        level, x, month = map(torch.tensor, self.lake_._inputs(train, origins, max(HORIZONS), observed=True))
        x_ar1 = torch.tensor(self.lake_._inputs(train, origins, max(HORIZONS), observed=False)[1][:, 1:])
        y = torch.tensor(np.column_stack([_future_change(train, h).loc[origins] for h in HORIZONS]))
        windows = self._windows(train, origins)
        self.net_ = nn.Sequential(nn.Conv1d(channels.shape[1], 8, 3), nn.ReLU(), nn.Conv1d(8, 8, 3), nn.ReLU(),
                                  nn.Flatten(), nn.Linear(8 * (self.window - 4), 3 * max(HORIZONS))).double()
        for w in self.net_[-1].parameters():  # start as the lake model's own forecast
            nn.init.zeros_(w)
        opt = torch.optim.AdamW(self.net_.parameters(), lr=self.lr, weight_decay=1e-2)

        def loss(rows):  # level error + weather error, each scaled by its own spread
            change, x_future = self._forward(windows[rows], level[rows], x[rows, 0], x_ar1[rows], month[rows])
            return (((change - y[rows]) / y.std(0)) ** 2).mean() + (((x_future - x[rows, 1:]) / self.xsd_) ** 2).mean()

        fit_rows, val_rows = slice(None, int(0.8 * len(origins))), slice(int(0.8 * len(origins)), None)
        best, state = loss(val_rows).item(), copy.deepcopy(self.net_.state_dict())
        for _ in range(self.epochs):
            opt.zero_grad()
            loss(fit_rows).backward()
            opt.step()
            with torch.no_grad():
                val = loss(val_rows).item()
            if val < best:
                best, state = val, copy.deepcopy(self.net_.state_dict())
        self.net_.load_state_dict(state)
        return self

    def _channels(self, df):
        return pd.concat([df[COMPONENTS + ["dlevel_mm", "level_m"]], make_features(df)[["month_sin", "month_cos"]]], axis=1)

    def _windows(self, df, origins):
        z = ((self._channels(df) - self.mu_) / self.sd_).to_numpy()
        return torch.tensor(z[df.index.get_indexer(origins)[:, None] + np.arange(1 - self.window, 1)]).transpose(1, 2)

    def _forward(self, windows, level, x_origin, x_ar1, month):
        p = torch.tensor(self.lake_.p_)
        x_future = x_ar1 + self.net_(windows).view(len(level), -1, 3) * self.xsd_
        supply = lake_supply(p, torch.cat([x_origin[:, None], x_future], 1), month)
        return (torch.stack(lake_route(p, level, supply), 1) - level[:, None]) * 1000, x_future

    def predict(self, df, origins, horizons):
        level, x, month = map(torch.tensor, self.lake_._inputs(df, origins, max(HORIZONS), observed=False))
        with torch.no_grad():
            change, _ = self._forward(self._windows(df, origins), level, x[:, 0], x[:, 1:], month)
        return change.numpy()[:, np.array(horizons) - 1]

**All models**, by the names used in every table and figure.

In [19]:
def all_models():
    return {
        "Persistence": Persistence(),
        "Seasonal climatology": SeasonalClimatology(),
        "Water balance": WaterBalance(),
        "Water balance + snowpack": SnowWaterBalance(),
        "Lake model": LakeModel(),
        "Lake model + snowpack": SnowLakeModel(),
        "Gradient boosting": GradientBoosting(),
        "Gradient boosting + snowpack": SnowGradientBoosting(),
        "Hybrid (WB + GB residual)": GradientBoosting(base=WaterBalance()),
        "Physics-guided CNN": PhysicsCNN(),
        "WB, perfect forcing (upper bound)": WaterBalance(perfect_forcing=True),
        "Lake model, perfect forcing (upper bound)": LakeModel(perfect_forcing=True),
    }

## 3. Evaluation

In [20]:
HORIZONS = [1, 2, 3, 4, 5, 6]
FIRST_TEST_YEAR = 1980  # leaves 30 years (1950-79) for the first training set

**Load the merged table.** Read `data/processed/monthly.csv` (written by
`data_processing.ipynb`) back in, with the months as the index.

In [21]:
def load_monthly():
    df = pd.read_csv(PROCESSED / "monthly.csv", index_col="month")
    df.index = pd.PeriodIndex(df.index, freq="M")
    return df

**Backtest.** Runs the rolling-origin test described above. The result has one row per
(model, target month, horizon) with the observed level, the predicted level and the
error in cm.

In [22]:
def backtest(df, first_test_year=FIRST_TEST_YEAR, verbose=True):
    rows = []
    for year in range(first_test_year, df.index.max().year + 1):
        train = df.loc[: f"{year - 1}-12"]
        targets = df.loc[f"{year}-01":f"{year}-12"].index
        models = {name: m.fit(train) for name, m in all_models().items()}
        for h in HORIZONS:
            origins = targets - h
            start = df.loc[origins, "level_m"].to_numpy()
            observed = df.loc[targets, "level_m"].to_numpy()
            for name, model in models.items():
                change_mm = model.predict(df, origins, [h])[:, 0]
                rows.append(pd.DataFrame({
                    "model": name, "target": targets, "origin": origins, "horizon": h,
                    "observed_m": observed, "predicted_m": start + change_mm / 1000.0,
                }))
        if verbose:
            print(f"  tested {year}", end="\r")
    preds = pd.concat(rows, ignore_index=True)
    preds["error_cm"] = (preds["predicted_m"] - preds["observed_m"]) * 100.0
    return preds

**RMSE table.** Summarise the backtest as root-mean-square error in cm, with models as
rows and horizons as columns.

In [23]:
def rmse_table(preds):
    rmse = preds.groupby(["model", "horizon"], sort=False)["error_cm"].apply(
        lambda e: np.sqrt(np.mean(e**2))
    )
    return rmse.unstack("horizon")

**Is a difference between two models real?** Resample whole test years many times
and see how much the RMSE gap moves. The result is
RMSE(a) − RMSE(b) per horizon with a 95% interval. Whole test years are resampled
together because errors within a year (and across overlapping multi-month forecasts) are
strongly correlated.

In [24]:
def bootstrap_rmse_diff(preds, model_a, model_b, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    for h in HORIZONS:
        sq = {}
        for name in (model_a, model_b):
            p = preds[(preds["model"] == name) & (preds["horizon"] == h)]
            sq[name] = (p["error_cm"] ** 2).groupby(p["target"].dt.year).agg(["sum", "count"])
        years = sq[model_a].index.to_numpy()
        a, b = sq[model_a].to_numpy(), sq[model_b].to_numpy()

        def diff(idx):
            return (np.sqrt(a[idx, 0].sum() / a[idx, 1].sum())
                    - np.sqrt(b[idx, 0].sum() / b[idx, 1].sum()))

        boots = [diff(rng.integers(0, len(years), len(years))) for _ in range(n_boot)]
        lo, hi = np.percentile(boots, [2.5, 97.5])
        out[h] = {"diff_cm": diff(np.arange(len(years))), "ci_low": lo, "ci_high": hi}
    return pd.DataFrame(out).T.rename_axis("horizon")

**Forecast ranges.** `TraceWaterBalance` is the water-balance model with the weather anomalies of one past year (`trace_year`) added to its AR(1) weather forecast, so the future weather keeps the month-to-month pattern of a real year. `ensemble_backtest` runs it once for every past year whose months end before the test year, and keeps the 2.5, 10, 50, 90 and 97.5% points of the results for every target month and horizon. With `model_error=True`, the errors the water-balance model makes on the training years even when the weather is known (its perfect-forcing errors) are added to every member as well, because the weather is not the only source of error. The design, including that choice, was fixed on 1950-1979, before any test year was looked at.

In [ ]:
class TraceWaterBalance(WaterBalance):
    """Water balance whose future weather is the AR(1) forecast plus the weather anomalies of one past year."""
    def _forcing(self, df, origins, k, observed):
        x = super()._forcing(df, origins, k, observed)
        past = origins + k - 12 * (origins.year - self.trace_year)  # the same months in the trace year
        return x + df.loc[past, COMPONENTS].to_numpy() - self.clim_.loc[past.month].to_numpy()


def ensemble_backtest(df, first_test_year=FIRST_TEST_YEAR, quantiles=(0.025, 0.1, 0.5, 0.9, 0.975), model_error=False):
    rows = []
    for year in range(first_test_year, df.index.max().year + 1):
        train = df.loc[: f"{year - 1}-12"]
        model, perfect = TraceWaterBalance().fit(train), WaterBalance(perfect_forcing=True).fit(train)
        targets = df.loc[f"{year}-01":f"{year}-12"].index
        for h in HORIZONS:
            origins = targets - h
            # every past year whose weather sequence for these months ends before the test year
            traces = [y for y in range(df.index.min().year + 1, year) if (origins + h - 12 * (origins.year - y)).max().year < year]
            members = []
            for y in traces:
                model.trace_year = y
                members.append(model.predict(df, origins, [h])[:, 0])
            members = np.array(members)
            if model_error:  # add the errors the model makes on the training years even when the weather is known
                o = train.index[:-h]
                err = _future_change(train, h).loc[o].to_numpy() - perfect.predict(train, o, [h])[:, 0]
                members = (members[:, None, :] + err[None, :, None]).reshape(-1, len(origins))
            level = df.loc[origins, "level_m"].to_numpy()[:, None] + np.quantile(members, quantiles, axis=0).T / 1000
            rows.append(pd.DataFrame(level, columns=[f"q{q}" for q in quantiles]).assign(
                target=targets, horizon=h, observed_m=df.loc[targets, "level_m"].to_numpy(), members=len(traces)))
    return pd.concat(rows, ignore_index=True)

## 4. Figures

In [25]:
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
BLUE, ORANGE, AQUA, VIOLET, GOLD, PINK, NAVY, BROWN, RUST = "#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#c99a06", "#d6336c", "#1d3f8f", "#8a5a00", "#a63d17"
MODEL_STYLE = {  # colour follows the model in every figure
    "Persistence": dict(color=MUTED, ls="-"),
    "Seasonal climatology": dict(color=VIOLET, ls="-"),
    "Water balance": dict(color=BLUE, ls="-"),
    "Water balance + snowpack": dict(color=NAVY, ls="-"),
    "Lake model": dict(color=GOLD, ls="-"),
    "Lake model + snowpack": dict(color=BROWN, ls="-"),
    "Gradient boosting": dict(color=ORANGE, ls="-"),
    "Gradient boosting + snowpack": dict(color=RUST, ls="-"),
    "Hybrid (WB + GB residual)": dict(color=AQUA, ls="-"),
    "Physics-guided CNN": dict(color=PINK, ls="-"),
    "WB, perfect forcing (upper bound)": dict(color=INK, ls="--"),
    "Lake model, perfect forcing (upper bound)": dict(color=GOLD, ls="--"),
}
MONTHS = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]

In [26]:
plt.rcParams.update({
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "lines.linewidth": 2, "legend.frameon": False, "figure.dpi": 100, "savefig.dpi": 200,
    "savefig.bbox": "tight",
})

**Saving a figure.** Write the figure to `figures/` as a PNG, close it, and print the file name.

In [27]:
def save(fig, name):
    fig.savefig(FIGURES / name)
    plt.close(fig)
    print("wrote figures/" + name)

**Figure 1.** The level record and the average seasonal cycle of supply.

In [28]:
def fig1_data(df):
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.4), width_ratios=[1.7, 1])
    t = df.index.to_timestamp()
    a.plot(t, df["level_m"], color=BLUE, lw=1.3)
    for label, month, dy in [("Record low\nJan 2013", "2013-01", -0.16),
                             ("Jul 2020 high", "2020-07", 0.1)]:
        p = pd.Period(month, "M")
        a.plot(p.to_timestamp(), df.at[p, "level_m"], "o", color=INK, ms=4)
        a.annotate(label, (p.to_timestamp(), df.at[p, "level_m"] + dy), ha="center",
                   va="center", fontsize=8, color=INK)
    a.set_title("Monthly mean level, Lake Michigan-Huron")
    a.set_ylabel("Level (m, IGLD 1985)")
    a.set_ylim(175.3, 177.7)

    clim = df[COMPONENTS].groupby(month_of(df.index)).mean()
    for col, label, color in [("precip_mm", "Precipitation", BLUE),
                              ("evap_mm", "Evaporation", ORANGE),
                              ("runoff_mm", "Runoff", AQUA)]:
        b.plot(range(1, 13), clim[col], color=color, marker="o", ms=3.5, label=label)
    b.set_xticks(range(1, 13), MONTHS)
    b.set_ylim(0, None)
    b.set_title("Average seasonal cycle of supply")
    b.set_ylabel("mm per month over the lake")
    b.legend(loc="upper center", ncol=3, fontsize=8, handlelength=1.2, columnspacing=1,
             bbox_to_anchor=(0.5, -0.12))
    save(fig, "fig1_data_overview.png")

**Figure 2.** Does supply explain level change? Observed level change against the two-month mean of net basin supply.

In [29]:
def fig2_water_balance(df):
    nbs2 = (df["nbs_mm"] + df["nbs_mm"].shift()) / 2
    ok = nbs2.notna()
    x, y = nbs2[ok].to_numpy(), df.loc[ok, "dlevel_mm"].to_numpy()
    slope, intercept = np.polyfit(x, y, 1)
    r = np.corrcoef(x, y)[0, 1]
    fig, a = plt.subplots(figsize=(5, 4))
    a.scatter(x, y, s=9, color=BLUE, alpha=0.45, lw=0)
    xs = np.array([x.min(), x.max()])
    a.plot(xs, intercept + slope * xs, color=INK, lw=1.5)
    a.axhline(0, color="#c3c2b7", lw=0.8)
    a.set_title("Level change follows net basin supply")
    a.set_xlabel("Net basin supply, mean of this and previous month (mm)")
    a.set_ylabel("Change in monthly mean level (mm)")
    a.text(0.03, 0.97, f"r = {r:.2f}\nslope = {slope:.2f}\nintercept = {intercept:.0f} mm",
           transform=a.transAxes, va="top", fontsize=8.5, color=INK)
    save(fig, "fig2_water_balance_check.png")
    return dict(r=r, slope=slope, intercept=intercept)

**Figure 3.** Forecast error by horizon for every model.

In [30]:
def fig3_rmse(preds):
    table = rmse_table(preds)
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.8))
    for name, style in MODEL_STYLE.items():
        a.plot(HORIZONS, table.loc[name], marker="o", ms=4, label=name, **style)
    a.set_title("Forecast error by horizon")
    a.set_xlabel("Forecast horizon (months ahead)")
    a.set_ylabel("RMSE of level (cm)")
    a.set_ylim(0, None)
    a.legend(fontsize=8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.2))

    ref = table.loc["Seasonal climatology"]
    for name in ["Water balance", "Water balance + snowpack", "Lake model", "Lake model + snowpack",
                 "Gradient boosting", "Gradient boosting + snowpack", "Hybrid (WB + GB residual)", "Physics-guided CNN"]:
        b.plot(HORIZONS, 100 * (1 - table.loc[name] / ref), marker="o", ms=4,
               label=name, **MODEL_STYLE[name])
    b.axhline(0, color=VIOLET, lw=1.5)
    b.text(6, 0.6, "seasonal climatology", color="#52514e", fontsize=8, ha="right", va="bottom")
    b.set_title("Error reduction vs. seasonal climatology")
    b.set_xlabel("Forecast horizon (months ahead)")
    b.set_ylabel("RMSE reduction (%)")
    b.legend(fontsize=8, loc="upper right")
    save(fig, "fig3_rmse_by_horizon.png")

In [31]:
def permutation_importance(df, split_year=2010, n_repeats=20, seed=0):
    rng = np.random.default_rng(seed)
    model = GradientBoosting().fit(df.loc[: f"{split_year - 1}-12"])
    feats = make_features(df)
    rows = {}
    for h in (1, 3, 6):
        targets = df.loc[f"{split_year}-01":].index
        origins = targets - h
        x = feats.loc[origins]
        y = (df.loc[targets, "level_m"].to_numpy() - df.loc[origins, "level_m"].to_numpy()) * 1000

        def rmse_cm(frame):
            return np.sqrt(np.mean((model.models_[h].predict(frame) - y) ** 2)) / 10

        base = rmse_cm(x)
        for group, cols in FEATURE_GROUPS.items():
            increases = []
            for _ in range(n_repeats):
                shuffled = x.copy()
                shuffled[cols] = x[cols].to_numpy()[rng.permutation(len(x))]
                increases.append(rmse_cm(shuffled) - base)
            rows[(h, group)] = np.mean(increases)
    return pd.Series(rows).unstack(0)

**Drawing Figure 4.** Bar chart of those RMSE increases at 1, 3 and 6 months ahead.

In [32]:
def fig4_importance(importance):
    order = importance[1].sort_values().index
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharey=True)
    for a, h in zip(axes, importance.columns):
        a.barh(order, importance.loc[order, h], color=ORANGE, height=0.6)
        a.set_title(f"{h} month{'s' if h > 1 else ''} ahead")
        a.set_xlabel("RMSE increase when shuffled (cm)")
        a.grid(axis="y", visible=False)
        a.axvline(0, color="#c3c2b7", lw=0.8)
    fig.suptitle("What the gradient-boosting model relies on (trained to 2009, scored on 2010-2023)",
                 x=0.01, ha="left", fontsize=10, fontweight="bold")
    fig.tight_layout()
    save(fig, "fig4_feature_importance.png")

**Figure 5.** Case study: the 2013 record low to the 2020 high.

In [33]:
def fig5_case_study(df, preds):
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.8), width_ratios=[1.4, 1])
    window = df.loc["2012-01":"2021-12"]
    a.plot(window.index.to_timestamp(), window["level_m"], color=INK, lw=2.2, label="Observed")
    for name in ["Persistence", "Water balance", "Gradient boosting"]:
        p = preds[(preds["model"] == name) & (preds["horizon"] == 6)
                  & (preds["target"] >= "2012-01") & (preds["target"] <= "2021-12")]
        a.plot(p["target"].dt.to_timestamp(), p["predicted_m"], lw=1.3,
               label=f"{name}, 6 months ahead", **MODEL_STYLE[name])
    a.set_title("2013 low to 2020 high: 6-month-ahead forecasts")
    a.set_ylabel("Level (m, IGLD 1985)")
    a.legend(fontsize=8, loc="upper left")

    # Cumulative supply anomalies relative to the pre-2013 climatology.
    base = df.loc[:"2012-12"]
    clim = base[COMPONENTS].groupby(month_of(base.index)).mean()
    rise = df.loc["2013-02":"2020-07"]
    anom = rise[COMPONENTS] - clim.loc[month_of(rise.index)].to_numpy()
    cum = anom.cumsum() / 10  # cm of water over the lake
    t = rise.index.to_timestamp()
    b.plot(t, cum["precip_mm"], color=BLUE, label="Precipitation")
    b.plot(t, -cum["evap_mm"], color=ORANGE, label="Evaporation (sign flipped)")
    b.plot(t, cum["runoff_mm"], color=AQUA, label="Runoff")
    b.axhline(0, color="#c3c2b7", lw=0.8)
    b.set_title("Cumulative supply anomaly, 2013-2020")
    b.set_ylabel("cm of water over the lake,\nvs. 1950-2012 average")
    b.legend(fontsize=8, loc="upper left")
    save(fig, "fig5_case_study_2013_2020.png")
    return {
        "level_rise_m": df.at[pd.Period("2020-07", "M"), "level_m"]
                        - df.at[pd.Period("2013-01", "M"), "level_m"],
        "cum_precip_cm": cum["precip_mm"].iloc[-1],
        "cum_minus_evap_cm": -cum["evap_mm"].iloc[-1],
        "cum_runoff_cm": cum["runoff_mm"].iloc[-1],
    }

**Figure 6.** The physics-guided CNN, drawn from the layer shapes of a trained network: a tensor of zeros is passed through the layers to read off each output shape, so the drawing always matches the code.

In [ ]:
def fig6_cnn(cnn):
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import FancyBboxPatch, Rectangle
    x, shapes, n_in = torch.zeros(1, cnn.net_[0].in_channels, cnn.window, dtype=torch.float64), [], cnn.net_[0].in_channels
    for layer in cnn.net_:
        x = layer(x)
        shapes.append(tuple(x.shape[1:]))
    fig, a = plt.subplots(figsize=(10, 4.2))
    a.set(xlim=(0, 100), ylim=(0, 33)), a.axis("off")
    sx, sy = 0.5, 1.2  # width and height of one cell of a grid
    for (rows, cols), x0, color in [((n_in, cnn.window), 4, BLUE), (shapes[0], 20.5, VIOLET), (shapes[2], 35.5, VIOLET)]:
        a.pcolormesh(x0 + np.arange(cols + 1) * sx, 17.5 - rows * sy / 2 + np.arange(rows + 1) * sy, np.zeros((rows, cols)),
                     cmap=ListedColormap([color]), edgecolors="white", linewidth=0.4, alpha=0.75)
    a.add_patch(Rectangle((4, 17.5 - n_in * sy / 2), 3 * sx, n_in * sy, fill=False, ec=INK, lw=1.2))
    a.plot([5.5, 20.75], [17.5 + n_in * sy / 2, 17.5 - shapes[0][0] * sy / 2 + sy / 2], color=INK, lw=0.8, ls=":")
    for i, name in enumerate(["P", "E", "R", "ΔL", "L", "sin", "cos"]):
        a.text(3.5, 17.5 - n_in * sy / 2 + (i + 0.5) * sy, name, ha="right", va="center", fontsize=6.5, color=INK)
    flat, out = np.linspace(12.5, 22.5, 9), np.linspace(4, 29, shapes[-1][0])
    for yf in flat:
        a.plot([50.5, 57.5], [np.full_like(out, yf), out], color=GRID, lw=0.4, zorder=0)
    a.scatter(np.full(len(flat), 50.5), flat, s=28, color=MUTED, zorder=2)
    a.text(50.5, 10.2, "⋮", ha="center", fontsize=10, color=MUTED)
    a.scatter(np.full(len(out), 57.5), out, s=28, zorder=2, color=[BLUE, ORANGE, AQUA] * (len(out) // 3))
    for k in range(6):
        a.text(58.8, out[3 * k + 1], f"+{k + 1}", va="center", fontsize=6.5, color=MUTED)
    for xt, text in [(10, f"Input\n{n_in} series × {cnn.window} months"), (26, f"Convolution 1\n{shapes[0][0]} filters, width {cnn.net_[0].kernel_size[0]}\n{shapes[0][0]} × {shapes[0][1]}"),
                     (40.5, f"Convolution 2\n{shapes[2][0]} × {shapes[2][1]}"), (50.5, f"Flatten\n{shapes[4][0]}")]:
        a.text(xt, 9.3, text, ha="center", va="top", fontsize=7.5, color=INK)
    a.text(57.5, 32.6, f"Linear: {shapes[5][0]} outputs", ha="center", va="top", fontsize=7.5, color=INK)
    for xt, name, color in [(54.5, "P", BLUE), (57.5, "E", ORANGE), (60.5, "R", AQUA)]:
        a.text(xt, 30.4, name, ha="center", fontsize=7.5, color=color, fontweight="bold")
    for y, text, color in [(25, "Corrected weather\nAR(1) forecast + $\\sigma_c\\,\\delta_{c,k}$", ORANGE),
                           (15, "Lake model\n(fixed physics, 10 steps a month)", GOLD), (5, "Forecast level change\n1 to 6 months ahead", AQUA)]:
        a.add_patch(FancyBboxPatch((70, y - 3), 27, 6, boxstyle="round,pad=0.3", fc=color, alpha=0.18, ec=color))
        a.text(83.5, y, text, ha="center", va="center", fontsize=8, color=INK)
    for y0, y1 in [(21.7, 18.3), (11.7, 8.3)]:
        a.annotate("", (83.5, y1), (83.5, y0), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1))
    a.annotate("", (69.5, 24), (60.5, 17), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1))
    a.annotate("", (59, 2.5), (69.5, 3.5), arrowprops=dict(arrowstyle="-|>", color=PINK, lw=1, ls="--"))
    a.text(2, 1.2, "Loss: level error + weather error; gradients flow back through the lake model (dashed)", fontsize=7, color=PINK)
    a.set_title("Physics-guided CNN: the network corrects the weather, the lake model makes the levels")
    save(fig, "fig6_cnn_architecture.png")

**Figure 7.** Forecast ranges: the 6-month-ahead ranges through the 2013 low and the 2020 high, and how often the observed level fell inside the 80% and 95% ranges at each horizon. Returns that coverage table.

In [ ]:
def fig7_ranges(ranges, df):
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.6), width_ratios=[1.6, 1])
    r = ranges[(ranges["horizon"] == 6) & (ranges["target"] >= "2012-01") & (ranges["target"] <= "2021-12")]
    t = r["target"].dt.to_timestamp()
    a.fill_between(t, r["q0.025"], r["q0.975"], color=BLUE, alpha=0.15, lw=0, label="95% range")
    a.fill_between(t, r["q0.1"], r["q0.9"], color=BLUE, alpha=0.3, lw=0, label="80% range")
    a.plot(t, r["q0.5"], color=BLUE, lw=1.3, label="Median")
    a.plot(df.loc["2012-01":"2021-12"].index.to_timestamp(), df.loc["2012-01":"2021-12", "level_m"], color=INK, lw=2, label="Observed")
    a.set_title("Water balance: 6-month-ahead ranges, 2012-2021")
    a.set_ylabel("Level (m, IGLD 1985)")
    a.legend(fontsize=8, loc="upper left")
    cover = 100 * pd.DataFrame({"80% range": ranges["observed_m"].between(ranges["q0.1"], ranges["q0.9"]),
                                "95% range": ranges["observed_m"].between(ranges["q0.025"], ranges["q0.975"])}).groupby(ranges["horizon"]).mean()
    for (name, nominal), color in zip([("80% range", 80), ("95% range", 95)], [BLUE, ORANGE]):
        b.plot(cover.index, cover[name], marker="o", ms=4, color=color, label=f"{name} (aim {nominal}%)")
        b.axhline(nominal, color=color, lw=1, ls="--")
    b.set(title="How often the observed level fell inside", xlabel="Forecast horizon (months ahead)",
          ylabel="Share of target months (%)", ylim=(50, 100))
    b.legend(fontsize=8, loc="lower right")
    save(fig, "fig7_forecast_ranges.png")
    return cover

**Figure 8.** The snowpack: its average seasonal cycle next to the runoff, and the snowpack at the end of March against the runoff of the following April to June, one point per year. Returns the correlation of the two.

In [ ]:
def fig8_snowpack(snow, df):
    pack, month = snow.snowpack(df) @ snow.p_[3:], month_of(df.index)  # mm over the lake, weighted like the water it releases
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.6))
    a.plot(range(1, 13), pack.groupby(month).mean(), color=VIOLET, marker="o", ms=3.5, label="Snowpack on the land (end of month)")
    a.plot(range(1, 13), df["runoff_mm"].groupby(month).mean(), color=AQUA, marker="o", ms=3.5, label="Runoff into the lake")
    a.set_xticks(range(1, 13), MONTHS)
    a.set(title="Snow on the land becomes spring runoff", ylabel="mm over the lake")
    a.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
    spring = (month >= 4) & (month <= 6)
    march, runoff = pack[month == 3].to_numpy(), df.loc[spring, "runoff_mm"].groupby(df.index.year[spring]).sum().to_numpy()
    r = np.corrcoef(march, runoff)[0, 1]
    b.scatter(march, runoff, s=14, color=VIOLET, alpha=0.7, lw=0)
    b.plot(np.sort(march), np.polyval(np.polyfit(march, runoff, 1), np.sort(march)), color=INK, lw=1.2)
    b.text(0.03, 0.97, f"r = {r:.2f}", transform=b.transAxes, va="top", fontsize=8.5, color=INK)
    b.set(title="End-of-March snowpack and April-June runoff", xlabel="Snowpack at the end of March (mm over the lake)",
          ylabel="April-June runoff (mm over the lake)")
    save(fig, "fig8_snowpack.png")
    return r

**Figure 9.** A diagram of the water balance of the lake: the flows in and out, which of them are in the data, and the snowpack model on the land.

In [ ]:
def fig9_lake_system():
    from matplotlib.patches import FancyBboxPatch, Rectangle
    fig, a = plt.subplots(figsize=(10, 4.4))
    a.set(xlim=(0, 100), ylim=(0, 44)), a.axis("off")
    a.add_patch(Rectangle((0, 6), 40, 9, color="#e3d7bf"))
    a.add_patch(Rectangle((40, 3), 42, 12, color=BLUE, alpha=0.3))
    a.add_patch(Rectangle((82, 6), 18, 9, color="#e3d7bf"))
    a.plot([40, 82], [15, 15], color=BLUE, lw=2.5)
    a.text(61, 8.5, "Lake Michigan-Huron\nlevel $L$ (monthly mean)", ha="center", va="center", fontsize=9, color=INK)
    arrow = lambda xy, xytext, color, ls="-": a.annotate("", xy, xytext, arrowprops=dict(arrowstyle="-|>", color=color, lw=1.8, ls=ls))
    arrow((51, 15.5), (51, 27), BLUE)
    a.text(51, 28, "Precipitation $P$", ha="center", fontsize=8.5, color=INK)
    arrow((71, 27), (71, 15.5), ORANGE)
    a.text(71, 28, "Evaporation $E$", ha="center", fontsize=8.5, color=INK)
    arrow((40, 11), (31, 11), AQUA)
    a.text(35.5, 12.2, "Runoff $R$", ha="center", fontsize=8.5, color=INK)
    arrow((44, 15.5), (44, 37), MUTED, "--")
    a.text(44, 38, "Inflow from Lake Superior\n(St. Marys River)", ha="center", fontsize=8, color=MUTED)
    arrow((97, 11), (82, 11), MUTED, "--")
    a.text(90, 16.5, "Outflow (St. Clair River),\ngrows with the level", ha="center", fontsize=8, color=MUTED)
    for (x, y, w, text, color) in [(2, 33, 26, "Snow and rain on the land\n(daily temperature, precipitation)", VIOLET),
                                   (2, 22.5, 26, "Snowpack: builds below 0 °C,\nmelts with degree-days", VIOLET),
                                   (2, 1, 26, "", None)]:
        if color:
            a.add_patch(FancyBboxPatch((x, y), w, 6.5, boxstyle="round,pad=0.3", fc=color, alpha=0.15, ec=color))
            a.text(x + w / 2, y + 3.25, text, ha="center", va="center", fontsize=8, color=INK)
    a.add_patch(FancyBboxPatch((2, 7.5), 26, 6, boxstyle="round,pad=0.3", fc=BROWN, alpha=0.15, ec=BROWN))
    a.text(15, 10.5, "Water in the land (store);\nevapotranspiration leaves it", ha="center", va="center", fontsize=8, color=INK)
    arrow((15, 29.3), (15, 32.7), VIOLET)
    arrow((15, 14), (15, 22.2), VIOLET)
    a.text(16, 18, "melt and rain", fontsize=7.5, color=MUTED)
    a.text(1, 1.5, "Solid arrows: flows in the data.  Dashed: not in the data.  Purple: the snowpack model.", fontsize=8, color=INK)
    a.text(99, 1.5, "net basin supply  NBS = P - E + R", ha="right", fontsize=8, color=INK)
    a.set_title("The water balance of Lake Michigan-Huron", pad=14)
    save(fig, "fig9_lake_system.png")

**Figure 10.** A diagram of the rolling-origin backtest.

In [ ]:
def fig10_backtest():
    fig, a = plt.subplots(figsize=(10, 2.7))
    for row, year in enumerate([2023, 2022, None, 1982, 1981, 1980]):
        if year is None:
            a.text(1985, row, "⋮", ha="center", va="center", fontsize=12, color=MUTED)
            continue
        a.barh(row, year - 1950, left=1950, height=0.6, color="#cfcdc3", label="Training years" if year == 1980 else None)
        a.barh(row, 1, left=year, height=0.6, color=ORANGE, label="Test year" if year == 1980 else None)
        a.text(1948.5, row, f"test {year}", ha="right", va="center", fontsize=8, color=INK)
    a.set(xlim=(1938, 2026), yticks=[], xlabel="Year")
    a.grid(axis="y", visible=False)
    a.legend(fontsize=8, loc="upper right", ncol=2)
    a.set_title("Rolling-origin backtest: each test year is forecast by models fitted only on earlier years")
    save(fig, "fig10_backtest.png")

**Figure 11.** One set of forecasts from a single origin (by default the end of March 2023, the last spring in the record): the observed level, five models and the 80% and 95% ranges of the water-balance model.

In [ ]:
def fig11_example(preds, ranges, df, origin="2023-03"):
    o, fig, a = pd.Period(origin, "M"), *plt.subplots(figsize=(10, 3.8))
    r = pd.concat([ranges[(ranges["horizon"] == h) & (ranges["target"] == o + h)] for h in HORIZONS])
    t, start = [o.to_timestamp()] + [p.to_timestamp() for p in r["target"]], df.at[o, "level_m"]
    a.fill_between(t, [start, *r["q0.025"]], [start, *r["q0.975"]], color=BLUE, alpha=0.15, lw=0, label="95% range (water balance)")
    a.fill_between(t, [start, *r["q0.1"]], [start, *r["q0.9"]], color=BLUE, alpha=0.3, lw=0, label="80% range (water balance)")
    for name in ["Persistence", "Seasonal climatology", "Water balance", "Water balance + snowpack", "Gradient boosting"]:
        p = preds[(preds["model"] == name) & (preds["origin"] == o)].sort_values("horizon")
        a.plot(t, [start, *p["predicted_m"]], lw=1.4, marker="o", ms=3, label=name, **MODEL_STYLE[name])
    w = df.loc[o - 12 : o + 6]
    a.plot(w.index.to_timestamp(), w["level_m"], color=INK, lw=2.4, label="Observed")
    a.axvline(o.to_timestamp(), color=MUTED, lw=1, ls=":")
    a.set(title=f"One set of forecasts, made at the end of {o.strftime('%B %Y')} for 1 to 6 months ahead", ylabel="Level (m, IGLD 1985)")
    a.legend(fontsize=7.5, loc="upper left", bbox_to_anchor=(1.01, 1))
    save(fig, "fig11_example_forecast.png")